# **Konvolüsyon, Görüntüleri Bulanıklaştırma ve Keskinleştirme**

#### **Bu derste şunları öğreneceğiz:**
1. Konvolüsyon İşlemleri
2. Bulanıklaştırma
3. Denoising
4. Keskinleştirme

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

# Define our imshow function 
def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()


### **Konvolüsyonlar Kullanarak Bulanıklaştırma**

In [ ]:
import cv2
import numpy as np

image = cv2.imread('../files/images/america6.jpg')
imshow('Original Image', image)

# 3 x 3 çekirdeğimizi oluşturuyoruz
kernel_3x3 = np.ones((3, 3), np.float32) / 9

# Çekirdeği bir görüntü ile birleştirmek için cv2.fitler2D kullanıyoruz
blurred = cv2.filter2D(image, -1, kernel_3x3)
imshow('3x3 Kernel Blurring', blurred)

# 9 x 9 çekirdeğimizi oluşturuyoruz
kernel_7x7 = np.ones((9, 9), np.float32) / 81

blurred2 = cv2.filter2D(image, -3, kernel_7x7)
imshow('9x9 Kernel Blurring', blurred2)

### **OpenCV'de yaygın olarak kullanılan diğer bulanıklaştırma yöntemleri**
- Düzenli Bulanıklaştırma
- Gauss Bulanıklaştırma
- Medyan Bulanıklaştırma

In [ ]:
import cv2
import numpy as np

image = cv2.imread('../files/images/america6.jpg')

# Ortalama alma işlemi, görüntüyü normalleştirilmiş bir kutu filtresi ile dönüştürerek yapılır.
# Bu, kutunun altındaki pikselleri alır ve merkezi öğeyi değiştirir
# Kutu boyutunun tek ve pozitif olması gerekir
blur = cv2.blur(image, (5,5))
imshow('Averaging', blur)

# Kutu filtresi yerine gauss çekirdeği
# Görüntüyü Gauss dağılımına göre ağırlıklandırarak bulanıklaştırır.
# Yani ortalama filtresinden (box filter) farklı olarak, komşu pikseller merkezden uzaklaştıkça daha az etkiye sahiptir.
# Bu sayede daha doğal bir yumuşatma sağlar ve gürültüyü azaltırken ayrıntıları korumada kutu filtresinden daha iyidir.
Gaussian = cv2.GaussianBlur(image, (5,5), 0)
imshow('Gaussian Blurring', Gaussian)

# Çekirdek alanı ve merkezi altındaki tüm piksellerin medyanını alır
# öğesi bu medyan değeri ile değiştirilir
# Gürültü azaltmada özellikle “tuz-biber (salt & pepper) gürültüsü” için çok etkilidir.
# Gaussian veya kutu filtresine göre kenarları daha iyi korur, çünkü aşırı yumuşatma yapmaz.
median = cv2.medianBlur(image, 5)
imshow('Median Blurring', median)

### **Bilateral Filtre**
#### dst = cv.bilateralFilter(src, d, sigmaColor, sigmaSpace, dst, borderType)
- **src** Kaynak 8 bit veya kayan noktalı, 1 kanallı veya 3 kanallı görüntü.
- **dst** src ile aynı boyutta ve türde hedef görüntü.
- **d** Filtreleme sırasında kullanılan her piksel komşuluğunun çapı. Pozitif değilse sigmaSpace'den hesaplanır.
- **sigmaColor** Renk uzayında filtre sigması. Parametrenin daha büyük bir değeri, piksel komşuluğundaki (bkz. sigmaSpace) daha uzak renklerin birbirine karıştırılacağı ve yarı eşit renkte daha geniş alanların elde edileceği anlamına gelir.
- **sigmaSpace** Koordinat uzayında filtre sigması. Parametrenin daha büyük bir değeri, renkleri yeterince yakın olduğu sürece daha uzak piksellerin birbirini etkileyeceği anlamına gelir (bkz. sigmaColor ). d>0 olduğunda, sigmaSpace'ten bağımsız olarak komşuluk boyutunu belirtir. Aksi takdirde, d sigmaSpace ile orantılıdır.
- **borderType** görüntünün dışındaki pikselleri ekstrapole etmek için kullanılan sınır modu




In [ ]:
# Bilateral, kenarları keskin tutarken gürültü gidermede çok etkilidir
bilateral = cv2.bilateralFilter(image, 9, 75, 75)
imshow('Bilateral Blurring', bilateral)

## **Gürültü Azaltma- Yerel Olmayan Araçlarla Gürültü Giderme**

Yerel araçlar (local filters): yalnızca yakın komşuluk penceresine bakar (blur, median).
Yerel olmayan araçlar (non-local means): görüntünün farklı yerlerindeki benzer desenleri de hesaba katar → daha akıllı bir gürültü azaltma sağlar.

**Yerel Olmayan Araçlarla Gürültü Azaltma 4 çeşidi vardır:**

- cv2.fastNlMeansDenoising() - tek bir gri tonlamalı görüntü ile çalışır
- cv2.fastNlMeansDenoisingColored() - renkli bir görüntü ile çalışır.
- cv2.fastNlMeansDenoisingMulti() - kısa sürede çekilen görüntü dizisiyle çalışır (gri tonlamalı görüntüler)
- cv2.fastNlMeansDenoisingColoredMulti() - yukarıdaki ile aynı, ancak renkli görüntüler için.


fastNlMeansDenoisingColored(InputArray src, OutputArray dst, float h=3, float hColor=3, int templateWindowSize=7, int searchWindowSize=21 )¶

#### fastNlMeansDenoisingColored için parametreler:

- **src** - Girdi 8 bit 3 kanallı görüntü.
- **dst** - src ile aynı boyut ve türde çıktı görüntüsü.
templateWindowSize - Ağırlıkları hesaplamak için kullanılan şablon ekinin piksel cinsinden boyutu. Tek olmalıdır. Önerilen değer 7 piksel
- **searchWindowSize** - Verilen piksel için ağırlıklı ortalamayı hesaplamak için kullanılan pencerenin piksel cinsinden boyutu. Tek olmalıdır. Performansı doğrusal olarak etkiler: daha büyük searchWindowsSize - daha büyük denoising süresi. Önerilen değer 21 piksel
- **h** - Parlaklık bileşeni için filtre gücünü düzenleyen parametre. Daha büyük h değeri gürültüyü mükemmel bir şekilde ortadan kaldırır ancak görüntü ayrıntılarını da ortadan kaldırır, daha küçük h değeri ayrıntıları korur ancak bir miktar gürültüyü de korur
- **hColor** - h ile aynıdır ancak renk bileşenleri içindir. Çoğu görüntü için 10'a eşit değer, renkli gürültüyü gidermek ve renkleri bozmamak için yeterli olacaktır




In [ ]:
image = cv2.imread('../files/images/america7.jpg')
imshow('Original', image)

dst = cv2.fastNlMeansDenoisingColored(image, None, 6, 6, 7, 21)
imshow('fastNlMeansDenoisingColored', dst)

### **Görüntüleri Keskinleştirme**

In [ ]:
# Resmimiz yükleniyor
image = cv2.imread('../files/images/america7.jpg')
imshow('Original', image)

# Şekillendirme çekirdeğimizi oluşturun, toplamının bir olması gerektiğini unutmayın
kernel_sharpening = np.array([[-1,-1,-1], 
                              [-1, 9,-1],
                              [-1,-1,-1]])

# görüntüye keskinleştirme çekirdeği uygulamak
sharpened = cv2.filter2D(image, -5, kernel_sharpening)
imshow('Sharpened Image', sharpened)